# Ficha técnica de una imagen propia

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Integrar todo el nivel de introducción sobre una fotografía propia: leerla con cuidado, describirla con una ficha técnica, preparar una versión de trabajo, anotarla, guardarla y convertir el procedimiento en un script que funcione sin Jupyter.

## Al terminar este cuaderno van a poder:

1. Cargar una imagen propia, con una alternativa preparada si el archivo no está disponible.
2. Comprobar si la lectura aplicó la orientación EXIF de la foto.
3. Construir una ficha técnica con datos del archivo, de la geometría y de la intensidad de la imagen.
4. Preparar una imagen de trabajo con un ancho fijo, conservando la proporción.
5. Anotar la ficha sobre la imagen y guardarla sin errores de color.
6. Convertir el procedimiento en un script autónomo y ejecutarlo desde el cuaderno.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Pipeline** | Una secuencia fija de pasos que transforma una entrada en un resultado. |
| **Ficha técnica** | Un resumen de datos medibles de una imagen: tamaño, peso, formato, brillo. |
| **Imagen de trabajo** | Una copia preparada para analizar, separada del archivo original. |
| **Imagen de respaldo** | Una imagen alternativa que el cuaderno usa si no encuentra la propia. |
| **Script** | Un archivo `.py` que se ejecuta de principio a fin sin abrir Jupyter. |
| **Argumento de línea de comandos** | Un dato que se le pasa a un script al ejecutarlo, como la ruta de la imagen. |

## De los ejercicios sueltos a un procedimiento

**Analogía.** En los cuadernos anteriores aprendimos a usar cada herramienta de un taller por separado. Este cuaderno es el primer mueble completo: las mismas herramientas, usadas en orden, con un plan y un resultado que se puede mostrar.

**Bajada tecnológica.** Cualquier proyecto que trabaja con muchas imágenes —un corpus de fotografías, un archivo de documentos escaneados, un conjunto de datos para entrenar un modelo— empieza con un paso como este: leer cada archivo de manera confiable, registrar sus datos básicos y dejar una versión de trabajo normalizada. Hacerlo a mano con cinco fotos es posible; con cinco mil, solo es posible con un script.

### ⚙ Configuración del entorno

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np

from utils import mostrar_imágenes

print(f"Python {sys.version_info.major}.{sys.version_info.minor} — OpenCV {cv2.__version__} — NumPy {np.__version__}")

## 1. Elegir la imagen

Para trabajar con una foto propia, cópienla dentro de esta carpeta (`001_intro`) y escriban su nombre en `RUTA_IMAGEN_PROPIA`. Si el archivo no existe, el cuaderno usa una imagen de ejemplo del curso, así puede ejecutarse completo en cualquier caso.

**Advertencia:** si la foto muestra personas que se pueden reconocer, trabajen con ella solo en su computadora y no la suban a un repositorio público sin su consentimiento.

In [ ]:
CARPETA_IMÁGENES = Path("..") / "imagenes"
CARPETA_SALIDAS = Path("salidas")
CARPETA_SALIDAS.mkdir(exist_ok=True)

# Si tienen una foto propia, cópienla a esta carpeta y escriban su nombre acá.
RUTA_IMAGEN_PROPIA = Path("mi_imagen.jpg")
RUTA_DE_RESPALDO = CARPETA_IMÁGENES / "small-boats_1.jpg"

if RUTA_IMAGEN_PROPIA.exists():
    ruta_elegida = RUTA_IMAGEN_PROPIA
else:
    ruta_elegida = RUTA_DE_RESPALDO

usa_imagen_propia = ruta_elegida == RUTA_IMAGEN_PROPIA
print(f"¿Usamos una imagen propia? {usa_imagen_propia} — archivo: {ruta_elegida}")

La decisión se toma con un `if` antes de leer, en lugar de intentar leer y reaccionar al error. Así el cuaderno nunca se detiene por un archivo que falta: si la foto propia no está, avisa y sigue con la de respaldo.

## 2. Leer con cuidado: la orientación EXIF

**Analogía.** Un celular apoyado de costado guarda la foto acostada y anota en una etiqueta "mostrar girada". Si quien la abre lee la etiqueta, la ve derecha; si la ignora, la ve de costado.

**Bajada tecnológica.** En `001_traduccion` vimos que `cv2.imread` aplica esa etiqueta EXIF por defecto. En un proyecto que mide ancho y alto de fotos de celulares distintos, saber si la orientación se aplicó cambia el resultado de la medición.

In [ ]:
imagen_bgr = cv2.imread(filename=str(ruta_elegida), flags=cv2.IMREAD_COLOR)
assert imagen_bgr is not None, f"No se pudo leer la imagen: {ruta_elegida}"

banderas_sin_giro = cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION
imagen_sin_giro = cv2.imread(filename=str(ruta_elegida), flags=banderas_sin_giro)

forma_con_exif = imagen_bgr.shape
forma_sin_exif = imagen_sin_giro.shape
se_aplicó_rotación = forma_con_exif != forma_sin_exif

print(f"Forma leída normalmente: {forma_con_exif}")
print(f"Forma ignorando EXIF:    {forma_sin_exif}")
print(f"¿La lectura giró la imagen? {se_aplicó_rotación}")

Si las dos formas coinciden, la foto no tenía una marca de orientación que cambie su tamaño, o no la necesitaba. Si difieren, OpenCV la giró durante la lectura: la forma "leída normalmente" es la que corresponde a cómo se ve la escena. La imagen de ejemplo del curso no trae orientación EXIF, así que en ese caso las dos formas son iguales.

## 3. Construir la ficha técnica

Una ficha técnica responde tres grupos de preguntas: **qué archivo es**, **qué forma tiene** y **cómo es su luz**. Las armamos por partes y al final las reunimos en un diccionario.

In [ ]:
imagen_rgb = cv2.cvtColor(src=imagen_bgr, code=cv2.COLOR_BGR2RGB)

# 'suffix' devuelve la extensión del archivo, por ejemplo '.jpg'.
extensión = ruta_elegida.suffix
formato = extensión.lower()
peso_en_kb = ruta_elegida.stat().st_size / 1024

print(f"Formato: {formato} | Peso: {peso_en_kb:.1f} KB")

El formato y el peso describen el archivo, no la escena. Recuerden `003_archivos`: un JPEG liviano ya perdió información en la compresión, y ningún paso posterior la va a recuperar.

In [ ]:
alto = imagen_rgb.shape[0]
ancho = imagen_rgb.shape[1]
canales = imagen_rgb.shape[2]

if ancho > alto:
    orientación = "horizontal"
elif ancho < alto:
    orientación = "vertical"
else:
    orientación = "cuadrada"

print(f"Ancho: {ancho} | Alto: {alto} | Canales: {canales} | Orientación: {orientación}")

La orientación sale de comparar ancho y alto. Es un dato que parece trivial, pero en un corpus de fotos de celular separa rápidamente las tomas verticales de las horizontales.

### ✎ Antes de ejecutar

La siguiente celda calcula el brillo medio y cuenta cuántos niveles de gris distintos tiene la imagen. Para su foto, ¿esperan un brillo medio más cerca de 0, de 128 o de 255? ¿Van a aparecer los 256 niveles posibles?

In [ ]:
imagen_gris = cv2.cvtColor(src=imagen_rgb, code=cv2.COLOR_RGB2GRAY)
brillo_medio = imagen_gris.mean()
niveles_presentes = np.unique(ar=imagen_gris)
cantidad_de_niveles = niveles_presentes.size

canales_rgb = cv2.split(m=imagen_rgb)
canal_rojo = canales_rgb[0]
canal_verde = canales_rgb[1]
canal_azul = canales_rgb[2]
media_rojo = canal_rojo.mean()
media_verde = canal_verde.mean()
media_azul = canal_azul.mean()
print(f"Brillo medio: {brillo_medio:.1f} | Niveles de gris distintos: {cantidad_de_niveles}")
print(f"Media por canal → R: {media_rojo:.1f} | G: {media_verde:.1f} | B: {media_azul:.1f}")

Tres lecturas posibles de estos números, válidas para cualquier foto:

- **Brillo medio.** Cerca de 0, la imagen es mayormente oscura; cerca de 255, mayormente clara. Un valor intermedio no garantiza buena exposición: puede ser la mezcla de zonas muy oscuras y muy claras.
- **Niveles distintos.** Si hay bastante menos de 256, la imagen no usa todo el rango: puede estar apagada, o haber pasado por una cuantización, como vimos en `005_resolucion`.
- **Media por canal.** Si un canal es claramente mayor que los otros dos, la escena tiene una dominante de ese color.

Reunimos todo en un diccionario: es la ficha técnica, el primer producto del cuaderno.

In [ ]:
ficha_técnica = {
    "archivo": ruta_elegida.name,
    "formato": formato,
    "peso_kb": round(peso_en_kb, 1),
    "ancho_x_alto": f"{ancho} x {alto}",
    "orientacion": orientación,
    "giro_exif": se_aplicó_rotación,
    "brillo_medio": round(brillo_medio, 1),
    "niveles_de_gris": cantidad_de_niveles,
}

for campo in ficha_técnica:
    valor = ficha_técnica[campo]
    print(f"{campo:<18}{valor}")

Los nombres de los campos van sin tildes a propósito: en la sección 5 los vamos a escribir sobre la imagen con `putText`, que no sabe dibujarlas.

### ✎ Para pensar

- ¿Qué campo de la ficha cambiaría si guardaran la misma foto como PNG? ¿Y cuáles quedarían iguales?
- Si dos fotos tienen el mismo brillo medio, ¿son igual de oscuras a la vista? Piensen en una foto de noche con un farol encendido.

### ✎ Completado

Agreguen a la ficha un campo con el canal dominante: el nombre del canal con mayor media.

In [ ]:
# ─── Completen los pasos faltantes ───────────────────────────────────────────
#
# TODO 1: comparen media_rojo, media_verde y media_azul con if y elif.
# TODO 2: guarden en 'canal_dominante' el texto "rojo", "verde" o "azul".
# TODO 3: agreguen el campo a la ficha y vuelvan a mostrarla.
#
# if media_rojo >= media_verde and media_rojo >= media_azul:
#     canal_dominante = "rojo"
# elif ...:
#     canal_dominante = ...
# else:
#     canal_dominante = ...
# ficha_técnica["canal_dominante"] = canal_dominante
# ─────────────────────────────────────────────────────────────────────────────

## 4. Preparar la imagen de trabajo

**Analogía.** Antes de pegar fotos en un álbum, se recortan y se llevan a un tamaño parecido. No se modifica la foto original: se prepara una copia pensada para ese uso.

**Bajada tecnológica.** Normalizar el tamaño es el primer paso de casi cualquier análisis de muchas imágenes: permite comparar medidas entre fotos y controla el tiempo de procesamiento. Es el mismo principio de `004_encuadre`, aplicado con un criterio fijo.

Describimos la región de interés como una caja `(x, y, ancho, alto)`, igual que en `006_anotacion`. Por defecto la caja abarca la imagen completa.

In [ ]:
región_de_interés = {"x": 0, "y": 0, "ancho": ancho, "alto": alto}

fila_desde = región_de_interés["y"]
fila_hasta = región_de_interés["y"] + región_de_interés["alto"]
columna_desde = región_de_interés["x"]
columna_hasta = región_de_interés["x"] + región_de_interés["ancho"]

# '.copy()' separa el recorte de la imagen original: lo que hagamos después no la modifica.
recorte_de_trabajo = imagen_rgb[fila_desde:fila_hasta, columna_desde:columna_hasta].copy()

forma_del_recorte = recorte_de_trabajo.shape
print(f"Forma del recorte de trabajo: {forma_del_recorte}")

Con la región por defecto, el recorte tiene la misma forma que la imagen completa. Ahora lo llevamos a un ancho fijo de 800 píxeles, conservando la proporción y eligiendo la interpolación según haya que achicar o agrandar.

In [ ]:
ANCHO_DE_TRABAJO = 800
alto_del_recorte = recorte_de_trabajo.shape[0]
ancho_del_recorte = recorte_de_trabajo.shape[1]
alto_de_trabajo = round(alto_del_recorte * ANCHO_DE_TRABAJO / ancho_del_recorte)

# Al achicar conviene promediar; al agrandar, interpolar con suavidad (ver 004_encuadre).
if ANCHO_DE_TRABAJO < ancho_del_recorte:
    interpolación = cv2.INTER_AREA
else:
    interpolación = cv2.INTER_CUBIC

imagen_de_trabajo = cv2.resize(src=recorte_de_trabajo, dsize=(ANCHO_DE_TRABAJO, alto_de_trabajo), interpolation=interpolación)
forma_de_trabajo = imagen_de_trabajo.shape
print(f"Imagen de trabajo: {forma_de_trabajo}")

Comparamos la imagen completa con la imagen de trabajo.

In [ ]:
paneles = [
    {"imagen": imagen_rgb, "título": f"Original: {ancho} × {alto}"},
    {"imagen": imagen_de_trabajo, "título": f"Imagen de trabajo: {ANCHO_DE_TRABAJO} × {alto_de_trabajo}"},
]
mostrar_imágenes(paneles, alto_de_figura=5)

Las dos se ven iguales en pantalla porque la figura ajusta su tamaño, pero los títulos muestran la diferencia: la imagen de trabajo tiene un ancho fijo y la misma proporción que la original.

### ✎ Modificación guiada

Cambien `región_de_interés` para quedarse solo con una parte de la foto —en la imagen de ejemplo, el bote— y vuelvan a ejecutar las tres celdas de esta sección. Antes, anoten qué valores de `x`, `y`, `ancho` y `alto` necesitan, recordando que `x` cuenta columnas e `y` cuenta filas.

## 5. Anotar la ficha y guardar

Escribimos la ficha sobre una copia de la imagen de trabajo, una línea por campo, con un fondo negro para que se lea sobre cualquier foto.

In [ ]:
imagen_anotada = imagen_de_trabajo.copy()
posición_vertical = 40

for campo in ficha_técnica:
    valor = ficha_técnica[campo]
    línea_de_texto = f"{campo}: {valor}"
    esquina_superior = (10, posición_vertical - 28)
    esquina_inferior = (430, posición_vertical + 10)
    cv2.rectangle(img=imagen_anotada, pt1=esquina_superior, pt2=esquina_inferior, color=(0, 0, 0), thickness=-1)
    cv2.putText(img=imagen_anotada, text=línea_de_texto, org=(20, posición_vertical), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.7, color=(255, 255, 255), thickness=2, lineType=cv2.LINE_AA)
    posición_vertical = posición_vertical + 38

paneles = [{"imagen": imagen_anotada, "título": "Imagen de trabajo con su ficha técnica"}]
mostrar_imágenes(paneles, alto_de_figura=7)

Cada campo de la ficha quedó escrito en una franja negra sobre el borde izquierdo de la foto. Si el nombre del archivo tiene tildes o eñes, en la imagen van a aparecer signos de pregunta: es la limitación de `putText` que vimos en `006_anotacion`.

Guardamos el resultado. La imagen está en RGB, así que la convertimos a BGR antes de `imwrite`.

In [ ]:
imagen_anotada_bgr = cv2.cvtColor(src=imagen_anotada, code=cv2.COLOR_RGB2BGR)
ruta_de_salida = CARPETA_SALIDAS / "ficha_desde_cuaderno.png"

se_guardó = cv2.imwrite(filename=str(ruta_de_salida), img=imagen_anotada_bgr)
assert se_guardó, "No se pudo guardar la imagen anotada."

print(f"Imagen anotada guardada en: {ruta_de_salida}")

El cuaderno ya produjo dos resultados: la ficha técnica y la imagen anotada. Pero todo depende de ejecutar las celdas en orden, dentro de Jupyter. El último paso es convertir este procedimiento en un script.

## 6. Del cuaderno al script

**Analogía.** Una receta que se cocinó una vez probando y anotando en los márgenes todavía no es una receta publicable. Para que otra persona la repita, hay que pasarla en limpio: ingredientes, pasos y resultado, sin los tanteos.

**Bajada tecnológica.** Los cuadernos sirven para explorar; los scripts, para repetir. Un script se puede ejecutar sobre cientos de imágenes, programar para que corra solo o integrar en una aplicación. Es el formato que van a necesitar cuando un procedimiento pase de un ejercicio a un proyecto.

El comando mágico `%%writefile` escribe el contenido de una celda en un archivo, en lugar de ejecutarlo. Con `%%writefile -a` se agrega contenido al final de un archivo que ya existe. Vamos a armar el script por partes. La primera parte es el encabezado, las importaciones y la carpeta de salidas.

In [ ]:
%%writefile ficha_imagen.py
# ────────────────────────────────────────────────
# ficha_imagen.py
# Genera la ficha técnica de una imagen y guarda una copia anotada.
# Uso: python ficha_imagen.py ruta/de/la/imagen.jpg
# ────────────────────────────────────────────────
import sys
from pathlib import Path

import cv2

CARPETA_SALIDAS = Path("salidas")

Jupyter responde `Writing ficha_imagen.py` (o `Overwriting`, si el cuaderno ya se había ejecutado antes): el archivo ya existe en esta carpeta. El encabezado documenta para qué sirve el script y cómo se usa, algo que en un cuaderno queda repartido entre muchas celdas de texto.

La segunda parte obtiene la ruta de la imagen desde la línea de comandos. `sys.argv` es la lista de palabras que se escribieron al ejecutar el script: la posición 0 es el nombre del script y la posición 1, la ruta de la imagen.

In [ ]:
%%writefile -a ficha_imagen.py


def obtener_ruta_de_la_imagen():
    """Lee la ruta de la imagen desde la línea de comandos o muestra cómo usar el script."""
    if len(sys.argv) < 2:
        print("Uso: python ficha_imagen.py ruta/de/la/imagen.jpg")
        sys.exit(1)
    ruta_de_la_imagen = Path(sys.argv[1])
    return ruta_de_la_imagen

A partir de ahora Jupyter responde `Appending to ficha_imagen.py`: cada celda agrega su parte al final del archivo. Si alguien ejecuta el script sin indicar una imagen, la función muestra cómo usarlo en lugar de fallar con un error difícil de entender.

La tercera parte es la función de lectura, la misma que usamos en todo el nivel.

In [ ]:
%%writefile -a ficha_imagen.py


def leer_imagen_rgb(ruta_de_la_imagen):
    """Lee una imagen con OpenCV, verifica la lectura y la devuelve en RGB."""
    imagen_leída = cv2.imread(filename=str(ruta_de_la_imagen), flags=cv2.IMREAD_COLOR)
    assert imagen_leída is not None, f"No se pudo leer: {ruta_de_la_imagen}"
    imagen_convertida = cv2.cvtColor(src=imagen_leída, code=cv2.COLOR_BGR2RGB)
    return imagen_convertida

Tener esta función dentro del script significa que el script no depende de ningún cuaderno ni de `utils.py`: se puede copiar a otra carpeta y sigue funcionando.

La cuarta parte construye la ficha, ahora como función.

In [ ]:
%%writefile -a ficha_imagen.py


def construir_ficha(ruta_de_la_imagen, imagen_rgb):
    """Devuelve un diccionario con los datos técnicos principales de una imagen ya leída."""
    imagen_gris = cv2.cvtColor(src=imagen_rgb, code=cv2.COLOR_RGB2GRAY)
    peso_en_kb = ruta_de_la_imagen.stat().st_size / 1024
    brillo_medio = imagen_gris.mean()
    ficha = {
        "archivo": ruta_de_la_imagen.name,
        "peso_kb": round(peso_en_kb, 1),
        "ancho_x_alto": f"{imagen_rgb.shape[1]} x {imagen_rgb.shape[0]}",
        "brillo_medio": round(brillo_medio, 1),
    }
    return ficha

Esta versión es más corta que la ficha de la sección 3: deja afuera el formato, la orientación, el giro EXIF y los niveles de gris, para que la función siga siendo fácil de leer. Agregar esos campos es un buen ejercicio para después del cuaderno.

La quinta parte prepara la imagen de trabajo con un ancho fijo, como en la sección 4.

In [ ]:
%%writefile -a ficha_imagen.py


def preparar_imagen_de_trabajo(imagen_rgb, ancho_objetivo):
    """Redimensiona la imagen a un ancho fijo conservando su proporción."""
    alto_original = imagen_rgb.shape[0]
    ancho_original = imagen_rgb.shape[1]
    alto_objetivo = round(alto_original * ancho_objetivo / ancho_original)
    if ancho_objetivo < ancho_original:
        interpolación = cv2.INTER_AREA
    else:
        interpolación = cv2.INTER_CUBIC
    imagen_de_trabajo = cv2.resize(src=imagen_rgb, dsize=(ancho_objetivo, alto_objetivo), interpolation=interpolación)
    return imagen_de_trabajo

Es la misma lógica de la sección 4, sin la región de interés: el script trabaja siempre con la imagen completa y la lleva al ancho que se le indique.

La sexta parte escribe la ficha sobre una copia de la imagen, como en la sección 5.

In [ ]:
%%writefile -a ficha_imagen.py


def anotar_ficha(imagen_rgb, ficha):
    """Devuelve una copia de la imagen con cada campo de la ficha escrito en una franja negra."""
    imagen_anotada = imagen_rgb.copy()
    posición_vertical = 40
    for campo in ficha:
        valor = ficha[campo]
        línea_de_texto = f"{campo}: {valor}"
        cv2.rectangle(img=imagen_anotada, pt1=(10, posición_vertical - 28), pt2=(430, posición_vertical + 10), color=(0, 0, 0), thickness=-1)
        cv2.putText(img=imagen_anotada, text=línea_de_texto, org=(20, posición_vertical), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.7, color=(255, 255, 255), thickness=2, lineType=cv2.LINE_AA)
        posición_vertical = posición_vertical + 38
    return imagen_anotada

La franja negra tiene un ancho fijo de 430 píxeles. Alcanza para los campos de esta ficha sobre una imagen de 800 píxeles de ancho; con nombres de archivo muy largos, el texto podría salirse de la franja.

La séptima parte guarda la imagen, convirtiéndola a BGR antes de `imwrite`.

In [ ]:
%%writefile -a ficha_imagen.py


def guardar_imagen_rgb(imagen_rgb, ruta_de_salida):
    """Convierte una imagen RGB a BGR y la guarda, creando la carpeta si hace falta."""
    ruta_de_salida.parent.mkdir(exist_ok=True)
    imagen_bgr = cv2.cvtColor(src=imagen_rgb, code=cv2.COLOR_RGB2BGR)
    se_guardó = cv2.imwrite(filename=str(ruta_de_salida), img=imagen_bgr)
    assert se_guardó, f"No se pudo guardar: {ruta_de_salida}"

La conversión a BGR quedó adentro de la función: quien use el script ya no puede olvidarse de hacerla, que era justamente el error que provocamos a propósito en `006_anotacion`.

La octava parte muestra la ficha en la terminal.

In [ ]:
%%writefile -a ficha_imagen.py


def mostrar_ficha(ficha):
    """Imprime cada campo de la ficha en una línea."""
    for campo in ficha:
        valor = ficha[campo]
        print(f"{campo:<18}{valor}")

`mostrar_ficha` usa el mismo formato de columnas que la celda de la sección 3, para que la salida del script y la del cuaderno se puedan comparar a simple vista.

La última parte es el bloque principal. Todo lo que está debajo de `if __name__ == "__main__":` se ejecuta solo cuando el archivo se corre como programa, y no cuando otro archivo lo importa para usar sus funciones.

In [ ]:
%%writefile -a ficha_imagen.py


if __name__ == "__main__":
    ruta_de_la_imagen = obtener_ruta_de_la_imagen()
    imagen_rgb = leer_imagen_rgb(ruta_de_la_imagen)
    ficha = construir_ficha(ruta_de_la_imagen=ruta_de_la_imagen, imagen_rgb=imagen_rgb)
    imagen_de_trabajo = preparar_imagen_de_trabajo(imagen_rgb=imagen_rgb, ancho_objetivo=800)
    imagen_anotada = anotar_ficha(imagen_rgb=imagen_de_trabajo, ficha=ficha)
    ruta_de_salida = CARPETA_SALIDAS / f"ficha_{ruta_de_la_imagen.stem}.png"
    guardar_imagen_rgb(imagen_rgb=imagen_anotada, ruta_de_salida=ruta_de_salida)
    mostrar_ficha(ficha=ficha)
    print(f"Imagen anotada guardada en: {ruta_de_salida}")

El bloque principal solo encadena las funciones anteriores, en el mismo orden que siguió el cuaderno: obtener la ruta, leer, construir la ficha, preparar, anotar, guardar y mostrar. El script está completo.

### ✎ Antes de ejecutar

Vamos a ejecutar el script sobre la misma imagen que usó el cuaderno. ¿Qué campos esperan ver en la terminal? ¿Sus valores van a coincidir con los de la ficha de la sección 3?

Lo ejecutamos desde el cuaderno con `!`, que manda el comando a la terminal. Usamos `sys.executable`, la ruta del Python de este entorno, para que el script encuentre OpenCV instalado.

In [ ]:
# Equivalente a abrir una terminal en esta carpeta y escribir:
#   python ficha_imagen.py ../imagenes/small-boats_1.jpg
!"{sys.executable}" ficha_imagen.py "{ruta_elegida}"

La terminal muestra una versión reducida de la ficha: los cuatro campos del script, con los mismos valores que calculamos paso a paso en la sección 3. Además, el script guardó su propia imagen anotada en la carpeta `salidas`, con un nombre que incluye el del archivo original. El procedimiento ya no depende del cuaderno: funciona con cualquier ruta que se le pase.

### ✎ Para pensar

- ¿Qué hace `if __name__ == "__main__"`? ¿Qué pasaría si otro archivo hiciera `from ficha_imagen import construir_ficha` y ese bloque no existiera?
- Este script ya corre sin Jupyter. ¿Qué necesitaríamos para que genere la ficha de todas las fotos de una carpeta, sin tener que escribir cada ruta a mano?

## Actividad de producción

In [ ]:
# ─── Espacio de práctica ─────────────────────────────────────────────────────
#
# Consigna: comparen dos fotos propias con el script.
#   1. Elijan dos fotos tomadas con el celular: una vertical y una horizontal.
#   2. Cópienlas a esta carpeta y ejecuten el script sobre cada una.
#   3. Abran las dos imágenes anotadas que quedaron en 'salidas'.
#   4. Escriban en una celda de texto qué campos de la ficha explican las
#      diferencias entre las dos fotos y qué diferencia visible no aparece
#      en ningún campo de la ficha.
#
# ─────────────────────────────────────────────────────────────────────────────

## Cierre del nivel de introducción

En este nivel pasamos de abrir una imagen con otra biblioteca a construir un procedimiento reproducible sobre fotos propias.

| Cuaderno | Idea central | Herramientas |
|---|---|---|
| `001_traduccion` | La imagen sigue siendo un arreglo; OpenCV usa BGR | `imread`, `cvtColor`, banderas de lectura |
| `002_canales` | Tres descripciones del color: RGB, grises, HSV | `split`, `merge`, `COLOR_RGB2HSV` |
| `003_archivos` | Guardar es decidir qué conservar | `imwrite`, calidad, `absdiff` |
| `004_encuadre` | Recortar, redimensionar y espejar sin inventar información | slicing, `resize`, `flip`, `rotate` |
| `005_resolucion` | Dos pérdidas: muestras y niveles | slicing con paso, cuantización, `calcHist` |
| `006_anotacion` | Marcar resultados sin errores de coordenadas ni de color | `rectangle`, `circle`, `putText` |
| `007_ficha` | Del procedimiento al script | `%%writefile`, `sys.argv` |

**Limitaciones de lo que hicimos.** La ficha describe la imagen con pocos números y no dice nada sobre su contenido: dos fotos muy distintas pueden tener fichas casi iguales. Tampoco modificamos la luz ni el color de la imagen: solo la medimos y la encuadramos.

Mejorar el contraste, limpiar el ruido, separar objetos del fondo y medir sus formas es justamente el trabajo del próximo nivel: **OpenCV fundamental**.